# Baseline XGBoost Model
Model trained strictly on non-spatial features.

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

df = pd.read_csv('../data/processed/spatial_panel.csv')

# Base Features only
features = [
    'population_2024', 'population_2020', 'gdp_usd_ppp_2014', 
    'aging_rate_pct', 'net_migration_rate', 'vacancy_rate_pct', 
    'dist_to_tokyo_km'
]
X = df[features]
y = df['target_pop_change_pct']

# Leave-One-Region-Out (Leave-One-Out approximation for N=47)
loo = LeaveOneOut()
y_true, y_pred = [], []

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Strong Regularization (Rules.md)
    model = xgb.XGBRegressor(
        max_depth=3, 
        learning_rate=0.05, 
        n_estimators=100,
        reg_alpha=0.5,
        reg_lambda=1.0,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred.append(model.predict(X_test)[0])
    y_true.append(y_test.values[0])

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
print(f"Baseline XGBoost - RMSE: {rmse:.4f}, R²: {r2:.4f}")

Baseline XGBoost - RMSE: 0.5445, R²: 0.8909
